#### Calculate RUSLE (A) from the Other Erosivity Factors [R*K*LS*C*P]... P is fixed as 1 bcos we did not identify data for Support Practice

In [1]:
# ===========================================
# RUSLE STACK aligned to LS grid (EPSG:26914; 10 m pixels)
# Inputs now:
#   K: EPSG:26914, 10 m
#   R: EPSG:26914, 10 m
#   LS: EPSG:26914, 10 m       <-- master grid
#   C: EPSG:4326,  ~8.983e-05 deg  (will be reprojected to LS)
#
# A = R * K * LS * C * P  [t·ha⁻¹·yr⁻¹]
# ===========================================

import rioxarray as rxr
import xarray as xr
import numpy as np
from rasterio.enums import Resampling

# ---------- USER PATHS ----------
ls_path     = r"mart_rusle_output/LS_factor.tif"           # EPSG:26914, 10 m
c_path      = r"mart_rusle_output/C_from_NDVI_water_built_zero_mart.tif"            # EPSG:4326, ~0.000089831528°
k_us_path   = r"mart_rusle_output/K_US_kffact_0_25cm_UTM14_mart.tif"            # EPSG:26914, 10 m (US units from gSSURGO)
r_path      = r"mart_rusle_output/R_factor_matched_to_K_mart.tif"            # EPSG:26914, 10 m (check units)
p_path      = None                                     # optional; if None -> 1 everywhere

out_A_path  = r"mart_rusle_output/A_t_ha_yr_utm14N_mart.tif"
out_dtype   = "float32"
out_nodata  = -9999.0

# ---------- UNIT FLAGS ----------
K_is_US = True   # True if K is in US customary (SSURGO Kffact/kwfact). Converts to SI with *0.1317
R_is_US = True   # True if R is in US customary. Converts to SI with *17.02

# ---------- HELPERS ----------
def same_grid(a, b):
    return (
        str(a.rio.crs) == str(b.rio.crs) and
        a.rio.transform() == b.rio.transform() and
        a.rio.width == b.rio.width and
        a.rio.height == b.rio.height
    )

def px_size(x):
    t = x.rio.transform()
    return (abs(t.a), abs(t.e))  # (xres, yres) as positive numbers

# ---------- LOAD MASTER (LS) ----------
LS = rxr.open_rasterio(ls_path, masked=True).squeeze("band", drop=True)
assert str(LS.rio.crs) == "EPSG:26914", f"LS CRS is {LS.rio.crs}, expected EPSG:26914"
# sanity: LS should be 10 m pixels (yres is negative for north-up)
print("[i] LS grid:", LS.rio.crs, "px=", px_size(LS))

# ---------- K (US→SI if needed); reproject only if grid differs ----------
K_raw = rxr.open_rasterio(k_us_path, masked=True).squeeze("band", drop=True)
K_si  = (K_raw * 0.1317) if K_is_US else K_raw
K = K_si if same_grid(K_si, LS) else K_si.rio.reproject_match(LS, resampling=Resampling.nearest)

# ---------- R (US→SI if needed); reproject only if grid differs ----------
R_raw = rxr.open_rasterio(r_path, masked=True).squeeze("band", drop=True)
R_si  = (R_raw * 17.02) if R_is_US else R_raw
# R is continuous → bilinear if we must reproject
R = R_si if same_grid(R_si, LS) else R_si.rio.reproject_match(LS, resampling=Resampling.bilinear)

# ---------- C: EPSG:4326 (deg) -> match LS (m), NEAREST to preserve masks/zeros ----------
C_src = rxr.open_rasterio(c_path, masked=True).squeeze("band", drop=True)
C = C_src.rio.reproject_match(LS, resampling=Resampling.nearest).clip(0, 1)

# ---------- P: load (NEAREST) or constant 1 ----------
if p_path:
    P_src = rxr.open_rasterio(p_path, masked=True).squeeze("band", drop=True)
    P = P_src if same_grid(P_src, LS) else P_src.rio.reproject_match(LS, resampling=Resampling.nearest)
else:
    P = xr.full_like(LS, 1.0)

# ---------- COMMON LAND MASK ----------
# Valid where LS, R, K exist; allow C==0 over bare areas; allow P to be 0 or 1
land_mask = (~np.isnan(LS)) & (~np.isnan(R)) & (~np.isnan(K))

# ---------- RUSLE ----------
A = (R * K * LS * C * P).where(land_mask)
A = A.astype(out_dtype)
A = A.rio.write_crs(LS.rio.crs, inplace=True)
A = A.rio.write_nodata(out_nodata, encoded=True)
A.rio.to_raster(out_A_path, compress="LZW")
print("✅ Wrote:", out_A_path)

# ---------- QA ----------
print("Grid match vs LS — R:", same_grid(R, LS), "K:", same_grid(K, LS), "C:", same_grid(C, LS), "P:", same_grid(P, LS))
print("Resolutions — R:", px_size(R), "K:", px_size(K), "C:", px_size(C), "LS:", px_size(LS))

def rminmax(x): 
    return float(np.nanmin(x.values)), float(np.nanmax(x.values))

print("Ranges — R(SI):[{:.1f},{:.1f}]  K(SI):[{:.4f},{:.4f}]  C:[{:.3f},{:.3f}]  LS:[{:.2f},{:.2f}]".format(
    *rminmax(R), *rminmax(K), *rminmax(C), *rminmax(LS)
))

for name, X in [('R', R), ('K', K), ('C', C), ('LS', LS), ('A', A)]:
    npx = X.size
    nnan = int(np.isnan(X.values).sum())
    print(f"{name}: NaN%={100.0*nnan/npx:.2f}")

[i] LS grid: EPSG:26914 px= (10.0, 10.0)
✅ Wrote: mart_rusle_output/A_t_ha_yr_utm14N_mart.tif
Grid match vs LS — R: True K: True C: True P: True
Resolutions — R: (10.0, 10.0) K: (10.0, 10.0) C: (10.0, 10.0) LS: (10.0, 10.0)
Ranges — R(SI):[5174.4,5428.6]  K(SI):[0.0000,0.0653]  C:[0.000,1.000]  LS:[0.00,106.84]
R: NaN%=1.65
K: NaN%=5.14
C: NaN%=5.16
LS: NaN%=0.09
A: NaN%=5.28


In [3]:
import numpy as np
v = A.values
print("A min/max:", float(np.nanmin(v)), float(np.nanmax(v)))
print("A p50/90/95/99/99.9:", np.nanpercentile(v,[50,90,95,99,99.9]))

A min/max: 0.0 1691.33349609375
A p50/90/95/99/99.9: [  3.85133982  19.70094986  27.98024712  52.76444931 112.69688391]


In [1]:
#### QA Checking results from RUSLE parameters and A (average soil erosion)

In [4]:
import numpy as np

def pct(a, ps): return {p: float(np.nanpercentile(a, p)) for p in ps}

print("C percentiles:", pct(C.values, [1,5,25,50,75,95,99]))
print("K(SI) percentiles:", pct(K.values, [1,5,25,50,75,95,99]))
print("LS percentiles:", pct(LS.values, [1,5,25,50,75,95,99,99.9]))
print("Pct pixels LS>20:", float(np.mean((LS.values>20) & ~np.isnan(LS.values)))*100)

v = A.values
print("A min/max:", float(np.nanmin(v)), float(np.nanmax(v)))
print("A p25/50/75/90/95/99/99.9:", np.nanpercentile(v,[25,50,75,90,95,99,99.9]))

C percentiles: {1: 0.0, 5: 0.00010038234455544233, 25: 0.0593506402300854, 50: 0.1418887108670272, 75: 0.3240679727741962, 95: 0.5588777996130022, 99: 0.6276736912473241}
K(SI) percentiles: {1: 0.03160799667239189, 5: 0.03160799667239189, 25: 0.03160799667239189, 50: 0.033715199679136276, 75: 0.0453048013150692, 95: 0.04872899875044823, 99: 0.05188979580998421}
LS percentiles: {1: 0.0, 5: 0.0, 25: 0.048147544264793396, 50: 0.13713973760604858, 75: 0.28849437832832336, 95: 0.756797194480896, 99: 1.5741766691207886, 99.9: 3.8950421810150146}
Pct pixels LS>20: 0.0017671008237045973
A min/max: 0.0 1691.33349609375
A p25/50/75/90/95/99/99.9: [  0.99062651   3.85133982  10.05587482  19.70094986  27.98024712
  52.76444931 112.69688391]


In [5]:
med_Rraw = float(np.nanmedian(R_raw))
print("R_raw median:", med_Rraw)

R_raw median: 311.08447265625


In [2]:
import numpy as np

# A range & percentiles (after your current compute)
finite = np.isfinite(A.values)
A_vals = A.values[finite]
qs = np.nanpercentile(A_vals, [50, 90, 95, 99, 99.9])
print("A min/max:", float(A_vals.min()), float(A_vals.max()))
print("A percentiles 50/90/95/99/99.9:", qs)

# How much area has extreme LS?
LS_qs = np.nanpercentile(LS.values, [95, 99, 99.9])
print("LS percentiles 95/99/99.9:", LS_qs)
pct_LS_gt20 = 100*np.nanmean((LS.values>20) & np.isfinite(LS.values))
print(f"Pct pixels with LS>20: {pct_LS_gt20:.2f}%")

A min/max: 0.0 1691.33349609375
A percentiles 50/90/95/99/99.9: [  3.85133982  19.70094986  27.98024712  52.76444931 112.69688391]
LS percentiles 95/99/99.9: [0.75679721 1.57417668 3.89512614]
Pct pixels with LS>20: 0.00%
